In [ ]:
%cd ../..
import os
import torch
import polars as pl
from tqdm import tqdm
from omegaconf import OmegaConf
import matplotlib.pyplot as plt

from dinov2.inference import generate_embeddings, build_model

In [ ]:
data_path = "/mnt/typhon/data/AI/DeepRDT/DeepRDT_rectum/Deep_rectum_crop"

metadata = pl.read_csv(os.path.join(data_path, "dfRectum_cropDetails_wClinic.csv"))
metadata.head()

In [ ]:
meta_mapids = metadata["MAPID"]
scan_mapids = [x.split(".pth")[0] for x in os.listdir(os.path.join(data_path, "torch_pth")) if x.endswith(".pth")]

print(len(meta_mapids), len(scan_mapids))

In [ ]:
def load_img(patient_id):
    hu_min, hu_max = -900, 700

    img_path = os.path.join(data_path, f"torch_pth/{patient_id}.pth")
    img = torch.load(img_path)

    hu = (img/255.0) * (hu_max - hu_min) + hu_min
    hu = torch.where(hu > -800, hu, -1000)

    return hu

In [ ]:
idx = 0
row = metadata.row(idx)
patient_id = row[0]
label = row[-1]
img = load_img(patient_id)

plt.imshow(img[20], cmap="gray")
plt.colorbar()
plt.show()

In [ ]:
config_path = "/home/48078029W/projects/radio-foundation/runs/base10pat/config.yaml"
checkpoint_path = "/home/48078029W/projects/radio-foundation/runs/base10pat/eval/training_99999/teacher_checkpoint.pth"

device = torch.device("cuda")

config = OmegaConf.load(config_path)
model, autocast_ctx = build_model(checkpoint_path, config, img_size=504, device=device)

In [ ]:
data_kwargs = dict(
    fmean = -573.8,
    fstd = 461.3,
    channels = 10,
    img_size = 504,
    patch_size = 14,
    device="cuda",
    block_size=64,
    no_crop=True,
    autocast_ctx=autocast_ctx,
)
output_path = "/scratch/VM/radio-foundation/cache/embeddings/DeepRDT"
os.makedirs(output_path, exist_ok=True)

for patient_id in tqdm(metadata["MAPID"], total=len(metadata)):
    img = load_img(patient_id)
    
    collated_features = generate_embeddings(
        img,
        model=model,
        **data_kwargs # type: ignore
    )

    output = {"cls": collated_features["cls"]}

    torch.save(output, os.path.join(output_path, f"{patient_id}.pth"))
